# Домашнее задание № 3. Исправление опечаток

## 1. Учет грамматики при оценке исправлений (3 балла)

В последнюю итерацию алгоритма для генерации исправлений добавьте еще один компонент - учет грамматической информации. Частично она уже учитывается за счет языковой модели (вероятность предсказывается для словоформы), но такой подход ограничен из-за того, что модель не может ничего предсказать для словоформ, которых не было в обучающей выборке. Чтобы это исправить постройте еще одну "языковую модель" на грамматических тэгах:
1) Используя mystem или pymorphy, разметьте какой-нибудь корпус (например, кусок wiki из семинара) или воспользуйтесь уже размеченным корпусом (например, opencorpora)
2) соберите униграмные и биграмные статистики на уровне грамматических тэгов (например, вместо `задача важна` у вас будет биграм `S,жен,неод=им,ед A=ед,кр,жен`). Для простоты можете начать только с частеречных тэгов и добавить остальную информацию позже
3) напишите функцию, которая будет оценивать вероятность данного предложения на основе грамматической языковой модели (статистик из предыдущего шага). Функция должна сначала преобразовать текст в грамматические тэги, используя точно такой же подход, что использовался на шаге 1. 
4) в функции correct_text_with_lm замените compute_sentence_proba на вашу новую функцию и прогоните получившийся алгоритм на данных
5) сравните предсказания с предсказанием изначального correct_text_with_lm, проверьте метрики и посмотрите на различие в ошибках и исправлениях, найдите несколько примеров отличий в предсказаниях этих подходов

In [5]:
def align_words(sent_1, sent_2):
    tokens_1 = sent_1.lower().split()
    tokens_2 = sent_2.lower().split()
    
    tokens_1 = [token.strip(punctuation) for token in tokens_1]
    tokens_2 = [token.strip(punctuation) for token in tokens_2]
    
    tokens_1 = [token for token in tokens_1 if token]
    tokens_2 = [token for token in tokens_2 if token]
    
    assert len(tokens_1) == len(tokens_2)
    
    return list(zip(tokens_1, tokens_2))

In [38]:
def ngrammer(tokens, n=2):
    """Создание n-грамм из списка токенов."""
    ngrams = []
    for i in range(0, len(tokens) - n + 1):
        ngrams.append(' '.join(tokens[i:i+n]))
    return ngrams

In [9]:
def normalize(text):
    normalized_text = [word.strip(punctuation) for word in text.split()]
    normalized_text = [word.lower() for word in normalized_text if word]
    return normalized_text

In [17]:
import pymorphy3
from collections import Counter
from nltk.tokenize import sent_tokenize
from string import punctuation
import numpy as np
from tqdm import tqdm


morph = pymorphy3.MorphAnalyzer()

In [13]:
IMPORTANT_GRAMMEMES = {
    # Части речи
    'POS': {'NOUN', 'ADJF', 'ADJS', 'VERB', 'INFN', 'PRTF', 'PRTS', 
            'GRND', 'NUMR', 'ADVB', 'NPRO', 'PRED', 'PREP', 'CONJ', 'PRCL', 'INTJ'},
    # Род
    'gender': {'masc', 'femn', 'neut'},
    # Число
    'number': {'sing', 'plur'},
    # Падеж
    'case': {'nomn', 'gent', 'datv', 'accs', 'ablt', 'loct', 'voct'},
    # Время
    'tense': {'past', 'pres', 'futr'},
    # Лицо
    'person': {'1per', '2per', '3per'},
}
POS_TAGS = {'NOUN', 'ADJF', 'ADJS', 'VERB', 'INFN', 'PRTF', 'PRTS', 
            'GRND', 'NUMR', 'ADVB', 'NPRO', 'PRED', 'PREP', 'CONJ', 'PRCL', 'INTJ'}

ALL_IMPORTANT = set()
for grammemes in IMPORTANT_GRAMMEMES.values():
    ALL_IMPORTANT.update(grammemes)


In [20]:

def get_simplified_tag(word):

    parsed = morph.parse(word)
    if not parsed:
        return '<UNK>'
    
    tag = parsed[0].tag
    parts = []

    if tag.POS:
        parts.append(str(tag.POS))
    
    if tag.gender:
        parts.append(str(tag.gender))

    if tag.number:
        parts.append(str(tag.number))
    
    if tag.case:
        parts.append(str(tag.case))
    
    if tag.tense:
        parts.append(str(tag.tense))

    if tag.person:
        parts.append(str(tag.person))
    
    return ','.join(parts) if parts else '<UNK>'


In [22]:

print("Пример преобразования в упрощённые тэги:")
examples = ["задача важна", "красивый дом"]

for example in examples:
    print(f"\n'{example}':")
    words = normalize(example)
    for word in words:
        full_tag = str(morph.parse(word)[0].tag)
        simple_tag = get_simplified_tag(word)
        print(f"  {word}: {full_tag} -> {simple_tag}")

Пример преобразования в упрощённые тэги:

'задача важна':
  задача: NOUN,inan,femn sing,nomn -> NOUN,femn,sing,nomn
  важна: ADJS,Qual femn,sing -> ADJS,femn,sing

'красивый дом':
  красивый: ADJF,Qual masc,sing,nomn -> ADJF,masc,sing,nomn
  дом: NOUN,inan,masc sing,nomn -> NOUN,masc,sing,nomn


In [ ]:
def collect_grammar_statistics(corpus_text):
    grammar_unigrams = Counter()
    grammar_bigrams = Counter()
    
    sentences = sent_tokenize(corpus_text)
    print(f"Обработка {len(sentences)} предложений...")
    
    for sentence in tqdm(sentences):
        words = normalize(sentence)
        if not words:
            continue
        
        tags = ['<START>']
        for word in words:
            tags.append(get_simplified_tag(word))
        tags.append('<END>')
        
        grammar_unigrams.update(tags)
        for i in range(len(tags) - 1):
            grammar_bigrams[f"{tags[i]} {tags[i+1]}"] += 1
    
    return grammar_unigrams, grammar_bigrams

In [40]:
vocab = open('/Users/kseniazavyalova/Downloads/data/wiki_data.txt', encoding='utf8').read()

In [ ]:

grammar_unigrams, grammar_bigrams = collect_grammar_statistics(vocab)

Обработка 203779 предложений...


100%|██████████| 203779/203779 [11:15<00:00, 301.84it/s] 


In [25]:
print(f"\nУникальных тэгов: {len(grammar_unigrams)}")
print(f"Уникальных биграмм: {len(grammar_bigrams)}")
print(f"\nТоп-10 тэгов:")
for tag, cnt in grammar_unigrams.most_common(10):
    print(f"  {tag}: {cnt}")


Уникальных тэгов: 237
Уникальных биграмм: 21583

Топ-10 тэгов:
  PREP: 625330
  <UNK>: 476595
  NOUN,masc,sing,gent: 288649
  NOUN,masc,sing,nomn: 277749
  CONJ: 258377
  <START>: 203695
  <END>: 203695
  NOUN,femn,sing,gent: 190539
  NOUN,femn,sing,nomn: 139972
  VERB,masc,sing,past: 125136


In [ ]:
def compute_grammar_sentence_proba(text):
    prob = 0.0
    tags = ['<START>'] + [get_simplified_tag(w) for w in normalize(text)] + ['<END>']
    
    for i in range(len(tags) - 1):
        bigram = f"{tags[i]} {tags[i+1]}"
        if tags[i] in grammar_unigrams and bigram in grammar_bigrams:
            prob += np.log(grammar_bigrams[bigram] / grammar_unigrams[tags[i]])
        else:
            prob += np.log(1e-5)
    return prob



In [27]:
print("\n" + "="*50)
print("ТЕСТ ГРАММАТИЧЕСКОЙ МОДЕЛИ")
print("="*50)
test_pairs = [
    ("красивый дом", "красивая дом"),
    ("большая собака", "большой собака"),
    ("новая машина", "новый машина"),
    ("он пришёл", "она пришёл"),
]

for good, bad in test_pairs:
    p_good = compute_grammar_sentence_proba(good)
    p_bad = compute_grammar_sentence_proba(bad)
    result = "✓ правильно" if p_good > p_bad else "✗ ошибка"
    print(f"\n'{good}' vs '{bad}'")
    print(f"  Тэги: {[get_simplified_tag(w) for w in normalize(good)]}")
    print(f"  vs:   {[get_simplified_tag(w) for w in normalize(bad)]}")
    print(f"  Prob: {p_good:.2f} vs {p_bad:.2f} -> {result}")


ТЕСТ ГРАММАТИЧЕСКОЙ МОДЕЛИ

'красивый дом' vs 'красивая дом'
  Тэги: ['ADJF,masc,sing,nomn', 'NOUN,masc,sing,nomn']
  vs:   ['ADJF,femn,sing,nomn', 'NOUN,masc,sing,nomn']
  Prob: -7.06 vs -10.75 -> ✓ правильно

'большая собака' vs 'большой собака'
  Тэги: ['ADJF,femn,sing,nomn', 'NOUN,femn,sing,nomn']
  vs:   ['ADJF,femn,sing,ablt', 'NOUN,femn,sing,nomn']
  Prob: -7.25 vs -15.43 -> ✓ правильно

'новая машина' vs 'новый машина'
  Тэги: ['ADJF,femn,sing,nomn', 'NOUN,femn,sing,nomn']
  vs:   ['ADJF,masc,sing,nomn', 'NOUN,femn,sing,nomn']
  Prob: -7.25 vs -10.26 -> ✓ правильно

'он пришёл' vs 'она пришёл'
  Тэги: ['NPRO,masc,sing,nomn,3per', 'VERB,masc,sing,past']
  vs:   ['NPRO,femn,sing,nomn,3per', 'VERB,masc,sing,past']
  Prob: -8.93 vs -16.85 -> ✓ правильно


In [42]:
def P(word):
    return vocab.get(word, 0)


In [43]:
def correction(word): 
    """Находим наиболее вероятное похожее слово."""
    return max(candidates(word), key=P)


def candidates(word): 
    """Генерируем кандидатов на исправление."""
    return known([word]) or known(edits1(word)) or known(edits2(word)) or [word]


def known(words): 
    """Выбираем слова, которые есть в корпусе."""
    return set(w for w in words if w in vocab)


def edits1(word):
    """Создаем кандидатов, которые отличаются на одну букву."""
    letters = 'йцукенгшщзхъфывапролджэячсмитьбюё'
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [L + R[1:] for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces = [L + c + R[1:] for L, R in splits if R for c in letters]
    inserts = [L + c + R for L, R in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)


def edits2(word): 
    """Создаем кандидатов, которые отличаются на две буквы."""
    return (e2 for e1 in edits1(word) for e2 in edits1(e1))


In [44]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_distances

In [45]:
import textdistance
import itertools

In [46]:
def get_closest_match_vec(text, X, vec, topn=20):
    """Поиск ближайших слов по векторному представлению."""
    v = vec.transform([text])
    similarities = cosine_distances(v, X)[0]
    topn_idx = similarities.argsort()[:topn] 
    return [(id2word[top], similarities[top]) for top in topn_idx]

In [37]:
def get_closest_match_with_metric(text, lookup, topn=20, metric=textdistance.levenshtein):
    """Поиск ближайших слов по метрике расстояния."""
    similarities = Counter()
    for word in lookup:
        similarities[word] = metric.similarity(text, word) 
    return similarities.most_common(topn)

In [36]:
def get_closest_hybrid_match(text, X, vec, topn=3, metric=textdistance.damerau_levenshtein):
    """Гибридный поиск: сначала векторный, потом по метрике."""
    candidates = get_closest_match_vec(text, X, vec, topn * 4)
    lookup = [cand[0] for cand in candidates]
    closest = get_closest_match_with_metric(text, lookup, topn, metric=metric)
    return closest


In [31]:
def predict_mistaken(word, vocab):
    """Определяет, является ли слово ошибочным (отсутствует в словаре)."""
    return word not in vocab

In [ ]:
def compute_sentence_proba(text, word_counts=None, bigram_counts=None):

    if word_counts is None:
        word_counts = unigrams
    if bigram_counts is None:
        bigram_counts = bigrams
        
    prob = 0
    tokens = normalize(text)
    for ngram in ngrammer(['<start>'] + tokens + ['<end>']):
        word1, word2 = ngram.split()
        if word1 in word_counts and ngram in bigram_counts:
            prob += np.log(bigram_counts[ngram] / word_counts[word1])
        else:
            prob += np.log(2e-5)
    return prob


In [28]:
def calculate_metrics(true, actual, predicted):
    """Вычисление метрик качества."""
    correct = 0
    total = 0
    total_mistaken = 0
    mistaken_fixed = 0
    total_correct = 0
    correct_broken = 0

    for i in range(len(true)):
        t, a, p = true[i], actual[i], predicted[i]
    
        if t == p:
            correct += 1
        total += 1
        
        if t == a:
            total_correct += 1
            if t != p:
                correct_broken += 1
        else:
            total_mistaken += 1
            if t == p:
                mistaken_fixed += 1

    return {
        "total_accuracy": correct/total,
        "fixed_mistakes": mistaken_fixed/total_mistaken if total_mistaken > 0 else 0,
        "broken_correct_words": correct_broken/total_correct if total_correct > 0 else 0
    }


In [ ]:
def correct_text_with_lm(text):

    sentence = normalize(text)
    corrections = [] 
    
    for word in sentence:
        if predict_mistaken(word, vocab):
            preds = get_closest_hybrid_match(word, X, vec)
            preds = [p[0] for p in preds]
            corrections.append(preds)
        else:
            corrections.append([word])

    possible_sentences = [" ".join(words) for words in itertools.product(*corrections)]
    most_prob_sentence = max(possible_sentences, key=lambda x: compute_sentence_proba(x))
    
    return most_prob_sentence

In [53]:
import re
corpus = open('/Users/kseniazavyalova/Downloads/data/wiki_data.txt', encoding='utf8').read()

bad = open('/Users/kseniazavyalova/Downloads/data/sents_with_mistakes.txt', encoding='utf8').read().splitlines()
true = open('/Users/kseniazavyalova/Downloads/data/correct_sents.txt', encoding='utf8').read().splitlines()

vocab = Counter(re.findall(r'\w+', corpus.lower()))

word2id = list(vocab.keys())
id2word = {i: word for i, word in enumerate(vocab)}

vec = CountVectorizer(analyzer='char', max_features=10000, ngram_range=(1, 3))
X = vec.fit_transform(vocab)

sentences = [['<start>'] + normalize(text) + ['<end>'] for text in sent_tokenize(corpus)]
unigrams = Counter()
bigrams = Counter()

for sentence in sentences:
    unigrams.update(sentence)
    bigrams.update(ngrammer(sentence))


In [48]:

print(f"Лексическая модель построена:")
print(f"  Словарь: {len(vocab)} слов")
print(f"  Униграммы: {len(unigrams)}")
print(f"  Биграммы: {len(bigrams)}")

Лексическая модель построена:
  Словарь: 368802 слов
  Униграммы: 532988
  Биграммы: 2692476


In [ ]:
def correct_text_with_grammar_lm(text):
    sentence = normalize(text)
    corrections = []
    
    for word in sentence:
        if predict_mistaken(word, vocab):
            preds = get_closest_hybrid_match(word, X, vec)
            corrections.append([p[0] for p in preds])
        else:
            corrections.append([word])

    possible = [" ".join(ws) for ws in itertools.product(*corrections)]
    return max(possible, key=compute_grammar_sentence_proba)


In [ ]:
def run_comparison():

    y_true = []
    y_actual = []
    preds_lexical = []
    preds_grammar = []
    

    cache_lexical = {}
    cache_grammar = {}
    

    differences = []

    for i in tqdm(range(len(true))):
        word_pairs = align_words(true[i], bad[i])
        
        for true_word, actual_word in word_pairs:
            y_true.append(true_word)
            y_actual.append(actual_word)
            
            if actual_word not in cache_lexical:
                cache_lexical[actual_word] = correction(actual_word)
            pred_lex = cache_lexical[actual_word]
            preds_lexical.append(pred_lex)

            if actual_word not in cache_grammar:
                cands = candidates(actual_word)
                if cands:

                    cache_grammar[actual_word] = max(cands, key=lambda w: (
                        grammar_unigrams.get(get_simplified_tag(w), 0),
                        vocab.get(w, 0)
                    ))
                else:
                    cache_grammar[actual_word] = actual_word
            pred_gram = cache_grammar[actual_word]
            preds_grammar.append(pred_gram)
            
            if pred_lex != pred_gram:
                differences.append({
                    'true': true_word,
                    'actual': actual_word,
                    'lexical': pred_lex,
                    'grammar': pred_gram,
                    'lex_correct': pred_lex == true_word,
                    'gram_correct': pred_gram == true_word
                })
    

    print("\n" + "="*70)
    print("МЕТРИКИ")
    print("="*70)
    print(f"{'Модель':<25} {'Accuracy':<12} {'Fixed':<12} {'Broken':<12}")
    print("-"*70)
    
    metrics_lex = calculate_metrics(y_true, y_actual, preds_lexical)
    metrics_gram = calculate_metrics(y_true, y_actual, preds_grammar)
    
    print(f"{'Лексическая (оригинал)':<25} {metrics_lex['total_accuracy']:<12.4f} "
          f"{metrics_lex['fixed_mistakes']:<12.4f} {metrics_lex['broken_correct_words']:<12.4f}")
    print(f"{'Грамматическая (новая)':<25} {metrics_gram['total_accuracy']:<12.4f} "
          f"{metrics_gram['fixed_mistakes']:<12.4f} {metrics_gram['broken_correct_words']:<12.4f}")
    
    
    print(f"\n{'='*70}")
    print(f"АНАЛИЗ РАЗЛИЧИЙ")
    print(f"{'='*70}")
    print(f"Всего случаев, где предсказания различаются: {len(differences)}")
    

    lex_wins = [d for d in differences if d['lex_correct'] and not d['gram_correct']]
    gram_wins = [d for d in differences if d['gram_correct'] and not d['lex_correct']]
    both_correct = [d for d in differences if d['lex_correct'] and d['gram_correct']]
    both_wrong = [d for d in differences if not d['lex_correct'] and not d['gram_correct']]
    
    print(f"\nСтатистика различий:")
    print(f"  Лексическая лучше:    {len(lex_wins)} случаев")
    print(f"  Грамматическая лучше: {len(gram_wins)} случаев")
    print(f"  Обе правильны:        {len(both_correct)} случаев")
    print(f"  Обе неправильны:      {len(both_wrong)} случаев")

    
    print(f"\n{'-'*70}")
    print("ПРИМЕРЫ: Грамматическая модель ЛУЧШЕ")
    print(f"{'-'*70}")
    for d in gram_wins[:10]:
        print(f"  Ошибка: '{d['actual']}' | Правильно: '{d['true']}'")
        print(f"    Лексическая:    '{d['lexical']}' ✗")
        print(f"    Грамматическая: '{d['grammar']}' ✓")
        print(f"    Тэг правильного: {get_simplified_tag(d['true'])}")
        print()
    
    print(f"{'-'*70}")
    print("ПРИМЕРЫ: Лексическая модель ЛУЧШЕ")
    print(f"{'-'*70}")
    for d in lex_wins[:10]:
        print(f"  Ошибка: '{d['actual']}' | Правильно: '{d['true']}'")
        print(f"    Лексическая:    '{d['lexical']}' ✓")
        print(f"    Грамматическая: '{d['grammar']}' ✗")
        print(f"    Тэг неправильного: {get_simplified_tag(d['grammar'])}")
        print()
    
    print(f"{'-'*70}")
    print("ПРИМЕРЫ: Обе модели ошиблись (но по-разному)")
    print(f"{'-'*70}")
    for d in both_wrong[:10]:
        print(f"  Ошибка: '{d['actual']}' | Правильно: '{d['true']}'")
        print(f"    Лексическая:    '{d['lexical']}' ✗")
        print(f"    Грамматическая: '{d['grammar']}' ✗")
        print()
    
    return {
        'y_true': y_true,
        'y_actual': y_actual,
        'preds_lexical': preds_lexical,
        'preds_grammar': preds_grammar,
        'metrics_lexical': metrics_lex,
        'metrics_grammar': metrics_gram,
        'differences': differences,
        'lex_wins': lex_wins,
        'gram_wins': gram_wins,
        'both_wrong': both_wrong
    }


results = run_comparison()

100%|██████████| 915/915 [04:22<00:00,  3.48it/s]


МЕТРИКИ
Модель                    Accuracy     Fixed        Broken      
----------------------------------------------------------------------
Лексическая (оригинал)    0.8708       0.5116       0.0760      
Грамматическая (новая)    0.8643       0.4612       0.0760      

АНАЛИЗ РАЗЛИЧИЙ
Всего случаев, где предсказания различаются: 529

Статистика различий:
  Лексическая лучше:    85 случаев
  Грамматическая лучше: 20 случаев
  Обе правильны:        0 случаев
  Обе неправильны:      424 случаев

----------------------------------------------------------------------
ПРИМЕРЫ: Грамматическая модель ЛУЧШЕ
----------------------------------------------------------------------
  Ошибка: 'сранно' | Правильно: 'странно'
    Лексическая:    'санно' ✗
    Грамматическая: 'странно' ✓
    Тэг правильного: ADVB

  Ошибка: 'пяный' | Правильно: 'пьяный'
    Лексическая:    'пятый' ✗
    Грамматическая: 'пьяный' ✓
    Тэг правильного: NOUN,masc,sing,nomn

  Ошибка: 'сначало' | Правильно: 'сначала'


## 2.  Symspell (5 баллов)

Реализуйте алгоритм Symspell. Он похож на алгоритм Норвига, но проще и быстрее. Он основан только на одной операции - удалении символа. Описание алгоритма по шагам:

1) Составляется словарь правильных слов  
2) На основе словаря правильных слов составляется словарь удалений - для каждого правильного слова создаются все варианты удалений и создается словарь, где ключ - слово с удалением, а значение - правильное слово  (обратите внимание, что для одного удаления может быть несколько правильных слов!) 
3) При исправлении слова с опечаткой сначала само слово проверятся по словарю удаления, а затем для этого слова генерируются все варианты удалений, и каждый вариант проверяется по словарю удалений. Если в словаре удалений таким образом находится совпадение, то соответствующее ему правильное слово становится исправлением.
Если совпадений несколько, то выбирается наиболее вероятное правильное слово  


Оцените качество полученного алгоритма теми же тремя метриками.

In [ ]:
from collections import defaultdict
from tqdm import tqdm

class SymSpell:
    def __init__(self, max_edit_distance=2):

        self.max_edit_distance = max_edit_distance
        self.vocabulary = {} 
        self.delete_dictionary = defaultdict(set)  
    
    def _generate_deletes(self, word, max_distance):

        deletes = set()
        
        def _recurse(current_word, remaining_distance):
            if remaining_distance == 0 or len(current_word) == 0:
                return
            
            for i in range(len(current_word)):
                delete = current_word[:i] + current_word[i+1:]
                if delete and delete not in deletes:
                    deletes.add(delete)
                    _recurse(delete, remaining_distance - 1)
        
        _recurse(word, max_distance)
        return deletes
    
    def build_dictionary(self, word_frequencies):

        self.vocabulary = word_frequencies.copy()
        
        print(f"Построение словаря удалений для {len(self.vocabulary)} слов...")
        
        for word in tqdm(self.vocabulary.keys()):
            self.delete_dictionary[word].add(word)
            
            deletes = self._generate_deletes(word, self.max_edit_distance)
            for delete in deletes:
                self.delete_dictionary[delete].add(word)
        
        print(f"Словарь удалений построен. Размер: {len(self.delete_dictionary)} записей")
        
        multi_match = sum(1 for v in self.delete_dictionary.values() if len(v) > 1)
        print(f"Удалений с несколькими правильными словами: {multi_match}")
    
    def correct(self, word):

        word = word.lower()
        
        if word in self.vocabulary:
            return word
        
        candidates = set()

        if word in self.delete_dictionary:
            candidates.update(self.delete_dictionary[word])
        
        deletes = self._generate_deletes(word, self.max_edit_distance)

        for delete in deletes:
            if delete in self.delete_dictionary:
                candidates.update(self.delete_dictionary[delete])
        
        if not candidates:
            return word
        
        return max(candidates, key=lambda w: self.vocabulary.get(w, 0))


In [61]:

symspell = SymSpell(max_edit_distance=2)

symspell.build_dictionary(vocab)


Построение словаря удалений для 368802 слов...


100%|██████████| 368802/368802 [02:07<00:00, 2902.27it/s] 


Словарь удалений построен. Размер: 14219365 записей
Удалений с несколькими правильными словами: 1456998


In [62]:
print("\n" + "="*60)
print("ПРИМЕРЫ РАБОТЫ SYMSPELL")
print("="*60)

test_words = ["привет", "превет", "прривет", "приввет", "молоко", "малако", "малоко"]
for word in test_words:
    corrected = symspell.correct(word)
    status = "✓" if word == corrected else f"-> {corrected}"
    print(f"  {word} {status}")


ПРИМЕРЫ РАБОТЫ SYMSPELL
  привет ✓
  превет -> первые
  прривет -> привело
  приввет -> привело
  молоко ✓
  малако -> мало
  малоко -> мало


In [ ]:
def evaluate_symspell():

    y_true = []
    y_actual = []
    y_preds = []
    
    cache = {}
    
    print("Оценка SymSpell на тестовых данных...")
    
    for i in tqdm(range(len(true))):
        word_pairs = align_words(true[i], bad[i])
        
        for true_word, actual_word in word_pairs:
            y_true.append(true_word)
            y_actual.append(actual_word)
            
            if actual_word not in cache:
                cache[actual_word] = symspell.correct(actual_word)
            
            y_preds.append(cache[actual_word])
    
    metrics = calculate_metrics(y_true, y_actual, y_preds)
    
    return y_true, y_actual, y_preds, metrics


y_true, y_actual, y_preds_symspell, metrics_symspell = evaluate_symspell()

print("\n" + "="*60)
print("МЕТРИКИ SYMSPELL")
print("="*60)
print(f"  Accuracy :     {metrics_symspell['total_accuracy']:.4f}")
print(f"  Fixed mistakes :   {metrics_symspell['fixed_mistakes']:.4f}")
print(f"  Broken correct :      {metrics_symspell['broken_correct_words']:.4f}")

Оценка SymSpell на тестовых данных...


100%|██████████| 915/915 [00:02<00:00, 342.33it/s]


МЕТРИКИ SYMSPELL
  Accuracy :     0.8252
  Fixed mistakes :   0.1817
  Broken correct :      0.0796


In [ ]:
def evaluate_norvig():

    y_true = []
    y_actual = []
    y_preds = []
    
    cache = {}
    
    print("Оценка алгоритма Норвига на тестовых данных...")
    
    for i in tqdm(range(len(true))):
        word_pairs = align_words(true[i], bad[i])
        
        for true_word, actual_word in word_pairs:
            y_true.append(true_word)
            y_actual.append(actual_word)
            
            if actual_word not in cache:
                cache[actual_word] = correction(actual_word)
            
            y_preds.append(cache[actual_word])
    
    metrics = calculate_metrics(y_true, y_actual, y_preds)
    
    return y_true, y_actual, y_preds, metrics


_, _, y_preds_norvig, metrics_norvig = evaluate_norvig()

# Сравнительная таблица
print("\n" + "="*70)
print("СРАВНЕНИЕ АЛГОРИТМОВ")
print("="*70)
print(f"{'Алгоритм':<20} {'Accuracy':<12} {'Fixed':<12} {'Broken':<12}")
print("-"*70)
print(f"{'Норвиг':<20} {metrics_norvig['total_accuracy']:<12.4f} "
      f"{metrics_norvig['fixed_mistakes']:<12.4f} {metrics_norvig['broken_correct_words']:<12.4f}")
print(f"{'SymSpell':<20} {metrics_symspell['total_accuracy']:<12.4f} "
      f"{metrics_symspell['fixed_mistakes']:<12.4f} {metrics_symspell['broken_correct_words']:<12.4f}")

Оценка алгоритма Норвига на тестовых данных...


100%|██████████| 915/915 [02:12<00:00,  6.89it/s]


СРАВНЕНИЕ АЛГОРИТМОВ
Алгоритм             Accuracy     Fixed        Broken      
----------------------------------------------------------------------
Норвиг               0.8708       0.5116       0.0760      
SymSpell             0.8252       0.1817       0.0796      


# Задание 3 (2 балла)

Используя любой из алгоритмов из семинара или домашки, детально проанализируйте получаемые ошибки. Улучшите алгоритм так, чтобы исправить ошибки. Улучшения в алгоритме должны быть общими, не привязанными к конкретным словам (например, словарь исключений не будет считаться). За каждое улучшение, которое исправляет 5+ ошибок вы получите 0.5 балла (максимум 2 в целом)

In [65]:
def collect_errors():
    """Собирает все ошибки базового алгоритма."""
    errors = []
    cache = {}
    
    for i in range(len(true)):
        word_pairs = align_words(true[i], bad[i])
        
        for true_word, actual_word in word_pairs:
            if actual_word not in cache:
                cache[actual_word] = correction(actual_word)
            predicted = cache[actual_word]
            
            if predicted != true_word:
                errors.append({
                    'true': true_word,
                    'actual': actual_word,
                    'predicted': predicted,
                    'was_mistake': actual_word != true_word,
                    'error_type': 'not_fixed' if predicted == actual_word else 'wrong_fix' if actual_word != true_word else 'broken'
                })
    
    return errors

errors = collect_errors()
print(f"Всего ошибок: {len(errors)}")
print(f"  Не исправлено: {len([e for e in errors if e['error_type'] == 'not_fixed'])}")
print(f"  Исправлено неправильно: {len([e for e in errors if e['error_type'] == 'wrong_fix'])}")
print(f"  Сломано правильное: {len([e for e in errors if e['error_type'] == 'broken'])}")

Всего ошибок: 1291
  Не исправлено: 237
  Исправлено неправильно: 392
  Сломано правильное: 662


In [70]:
import re
import numpy as np
from collections import Counter
from string import punctuation
from tqdm import tqdm

def prefix_match(word1, word2):
    """
    ИДЕЯ:
    Если человек начал писать слово правильно, а ошибся ближе к концу, то правильное исправление должно иметь то же начало.

    ПРИМЕР:
    Опечатка: "преключение" (вместо "приключение")
  
    Кандидаты на исправление:
    - "приключение" (префикс с опечаткой: "пр" = 2 символа)
    - "заключение"  (префикс с опечаткой: "" = 0 символов)
    """
    prefix = 0
    for c1, c2 in zip(word1, word2):
        if c1 == c2:
            prefix += 1
        else:
            break
    return prefix

def fix_repeats(word):
    """
    Схлопывает повторы символов (3+ подряд) до 1 или 2.
    
    Примеры:
      "ураааа" -> кандидаты {"ура", "ураа"} (если есть в словаре)
      "спасиииибо" -> {"спасибо"}
      "класссно" -> {"классно", "класно"}

    """
    candidates = set()
    
    reduced_1 = re.sub(r'(.)\1{2,}', r'\1', word)
    if reduced_1 in vocab:
        candidates.add(reduced_1)
    
    reduced_2 = re.sub(r'(.)\1{2,}', r'\1\1', word)
    if reduced_2 in vocab:
        candidates.add(reduced_2)
    
    return candidates


def correction_improved(word):
    word = word.lower()
    
    if word in vocab:
        return word
    
    repeat_candidates = fix_repeats(word)
    if repeat_candidates:
        return max(repeat_candidates, key=lambda w: vocab.get(w, 0))
    
    cands = candidates(word)
    
    if not cands:
        return word
    
    def score(candidate):
        freq = np.log(vocab.get(candidate, 1) + 1)
        pref = prefix_match(word, candidate) / max(len(word), 1)
        return 0.6 * freq + 0.4 * pref * 10
    
    return max(cands, key=score)


def evaluate_correction_function(correct_func, name=""):
    y_true_list = []
    y_actual_list = []
    y_preds = []
    cache = {}
    
    for i in range(len(true)):
        word_pairs = align_words(true[i], bad[i])
        for true_word, actual_word in word_pairs:
            y_true_list.append(true_word)
            y_actual_list.append(actual_word)
            
            if actual_word not in cache:
                cache[actual_word] = correct_func(actual_word)
            y_preds.append(cache[actual_word])
    
    metrics = calculate_metrics(y_true_list, y_actual_list, y_preds)
    return y_true_list, y_actual_list, y_preds, metrics

y_true_list, y_actual_list, y_preds_base, metrics_base = evaluate_correction_function(correction, "Базовый")
_, _, y_preds_improved, metrics_improved = evaluate_correction_function(correction_improved, "Улучшенный")


fixed = sum(1 for i in range(len(y_true_list)) 
            if y_preds_base[i] != y_true_list[i] and y_preds_improved[i] == y_true_list[i])
broken = sum(1 for i in range(len(y_true_list)) 
             if y_preds_base[i] == y_true_list[i] and y_preds_improved[i] != y_true_list[i])


print(f"{'+ Префикс + Повторы':<35} {metrics_improved['total_accuracy']:<12.4f} "
      f"{metrics_improved['fixed_mistakes']:<12.4f} {metrics_improved['broken_correct_words']:<12.4f}")
print("-"*70)
print(f"Исправлено ошибок базового алгоритма: {fixed}")
print(f"Сломано правильных ответов: {broken}")
print(f"Чистый выигрыш: {fixed - broken}")


print("\n--- Исправлено благодаря удалению повторов: ---")
repeat_examples = []
for i in range(len(y_true_list)):
    if y_preds_base[i] != y_true_list[i] and y_preds_improved[i] == y_true_list[i]:
        if re.search(r'(.)\1{2,}', y_actual_list[i]):
            repeat_examples.append({
                'actual': y_actual_list[i],
                'base': y_preds_base[i],
                'improved': y_preds_improved[i]
            })

for e in repeat_examples[:10]:
    print(f"  '{e['actual']}': базовый -> '{e['base']}' , улучшенный -> '{e['improved']}' ")

print("\n--- Исправлено благодаря учёту начала: ---")
prefix_examples = []
for i in range(len(y_true_list)):
    if y_preds_base[i] != y_true_list[i] and y_preds_improved[i] == y_true_list[i]:
        if not re.search(r'(.)\1{2,}', y_actual_list[i]):
            prefix_base = prefix_match(y_actual_list[i], y_preds_base[i])
            prefix_true = prefix_match(y_actual_list[i], y_true_list[i])
            if prefix_true > prefix_base:
                prefix_examples.append({
                    'actual': y_actual_list[i],
                    'true': y_true_list[i],
                    'base': y_preds_base[i],
                    'prefix_base': prefix_base,
                    'prefix_true': prefix_true
                })

for e in prefix_examples[:10]:
    print(f"  '{e['actual']}': базовый -> '{e['base']}', правильно: '{e['true']}'")

+ Префикс + Повторы                 0.8741       0.5373       0.0760      
----------------------------------------------------------------------
Исправлено ошибок базового алгоритма: 45
Сломано правильных ответов: 12
Чистый выигрыш: 33

--- Исправлено благодаря удалению повторов: ---
  'оччччень': базовый -> 'оччччень' , улучшенный -> 'очень' 
  'каааак': базовый -> 'казак' , улучшенный -> 'как' 
  'оооочень': базовый -> 'сорочень' , улучшенный -> 'очень' 
  'оооочень': базовый -> 'сорочень' , улучшенный -> 'очень' 
  'страааашно': базовый -> 'страааашно' , улучшенный -> 'страшно' 
  'оооочень': базовый -> 'сорочень' , улучшенный -> 'очень' 
  'оооочень': базовый -> 'сорочень' , улучшенный -> 'очень' 
  'оооочень': базовый -> 'сорочень' , улучшенный -> 'очень' 
  'ээээххх': базовый -> 'ээээххх' , улучшенный -> 'эх' 
  'ооооочень': базовый -> 'ооооочень' , улучшенный -> 'очень' 

--- Исправлено благодаря учёту начала: ---
  'зуные': базовый -> 'зные', правильно: 'зубные'
  'сначало': б